# 08 · Evaluate students against the ground truth

Scores trained students (notebook 07 runs) against the **evaluation ground truth** (built from KITTI-360's accumulated 3D semantics and boxes, never from the scan or the teachers), with `ocgf/evaluation/metrics.py`:

- **false-free rate**, both versions: `ffr_pred` = share of cells called free that are occupied, obstacle or unseen; `ffr_hazard` = share of those hazard cells that are called free;
- free-road IoU / precision / recall (and the road head alone), occupancy and unknown IoU;
- per distance band and in the **near-object band** (within 2 m of a car, large vehicle, pedestrian or cyclist box);
- per class: the share of box cells called free; calibration (ECE, reliability bins) and risk-coverage.

The model sees only the image, the scan and the calibration (its BEV inputs are recomputed from the scan).

Starter phase: drive 0000's **val** part of `kitti360_d0000_dev_v1` (debugging only, never paper results). Its ground truth is built here for the frames of `PART`, unless an evaluation-GT dataset is attached. `PART = "train"` scores the frames the model trained on: a check that the notebook works, never a result.

## Before you run

1. **Settings → Accelerator:** GPU T4 (CPU works, slower). **Internet:** On (code clone, DINOv2 weights).
2. **Add Input:** the output of each notebook-07 run in `RUNS`; drive 0000's data-prep output; the KITTI-360 starter (it also holds drive 0000's 3D semantics); `data-3d-bboxes`, to build the ground truth; for other drives **KITTI-360 3d semantics** (`greatgamedota/kitti360-3d-semantics`); and for object runs or `POST_HOC`, **`ocgf-mtrans-objects-v1`** (notebook 04).
3. **Save Version → Save & Run All.**

In [ ]:
from pathlib import Path

REPO_URL  = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF   = "p0/skeleton-geometry"       # branch, tag or commit (main once merged); pin a commit for the final run
RUNS      = ["dev_e0_concat_s0"]         # notebook-07 experiment folders (attached as inputs)
POST_HOC  = []                           # runs also scored with detected boxes subtracted (E1 baseline: "dev_e1_posthoc_s0")
PERTURBATIONS = []                       # E5, e.g. ["rings=2", "rings=4", "points=0.5", "box_drop=1.0", "blur=2"]
SPLIT     = "configs/splits/kitti360_d0000_dev_v1.json"
PART      = "val"                        # "test" only for final numbers; "train" only to check the pipeline
DRIVES    = ["2013_05_28_drive_0000_sync"]
QA        = 6                            # QA figures per run
SAVE_PREDS = False                       # also keep per-frame probabilities (calibration plots, failure cases)
BACKBONE  = None                         # None: the run's own backbone
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/eval")
GT_DIR    = Path("/kaggle/temp/eval_gt")  # ground truth built here when none is attached
STAGE_DIR = Path("/kaggle/temp/stage")
THREADS   = 32
CODE_DIR  = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())

## Runs and ground truth

Finds each run's checkpoint among the inputs, and the ground truth of each drive: an attached evaluation-GT dataset, or built here for the frames of `PART`.

In [ ]:
import json
from ocgf.datasets.locate import find_outputs, find_runs

run_dirs = {}
for name in RUNS:
    hits = find_runs(INPUT_ROOT, name)
    assert hits, f"run {name} not found: attach its notebook-07 output (Add Input)"
    progress = json.loads((hits[0] / "ckpt" / "progress.json").read_text())
    run_dirs[name] = hits[0]
    print(f"{name}: {hits[0]} ({progress['epoch'] + 1} of {progress['epochs']} epochs)")
    for other in hits[1:]:
        print(f"  WARNING: {name} is also attached at {other} (not used): attach one copy of each run")

def attached_gt(seq):
    found = find_outputs(INPUT_ROOT, seq, "kind", "eval_gt")
    return found[0].parent if found else None

GT_ROOTS = {}
for seq in DRIVES:
    root = attached_gt(seq)
    if root is None:
        sh(f'{sys.executable} scripts/build_eval_gt.py --input-root "{INPUT_ROOT}" --sequence {seq} --out "{GT_DIR}" '
           f'--split "{CODE_DIR / SPLIT}" --parts {PART} --qa 2 --stage-dir "{STAGE_DIR / "semantics"}" '
           f'--threads {THREADS}', cwd=CODE_DIR)
        root = GT_DIR
    GT_ROOTS[seq] = root
    print(f"{seq}: ground truth in {root}")
assert len(set(GT_ROOTS.values())) == 1, "keep every drive's ground truth under one folder"
EVAL_GT = next(iter(GT_ROOTS.values()))

## Evaluate

One run at a time, on the same frames. Runs in `POST_HOC` are evaluated a second time with the detected boxes subtracted after prediction (E1's baseline arm), listed as `<run>+posthoc`. Every evaluation is repeated under each test-time perturbation in `PERTURBATIONS` (E5), listed as `<run>@<perturbation>`.

In [ ]:
from ocgf.evaluation.perturb import parse_spec, spec_name

for spec in PERTURBATIONS:
    parse_spec(spec)                                     # fail early on a typo
EVALS = [(name, ph, spec) for name, ph in [(n, False) for n in run_dirs] + [(n, True) for n in POST_HOC if n in run_dirs]
         for spec in [""] + PERTURBATIONS]
label = lambda name, ph, spec: name + ("+posthoc" if ph else "") + (f"@{spec}" if spec else "")
folder = lambda name, ph, spec: OUT_DIR / (name + ("+posthoc" if ph else "")) / PART / spec_name(spec)
for name, ph, spec in EVALS:
    sh(f'{sys.executable} scripts/evaluate_student.py --run "{run_dirs[name]}" --input-root "{INPUT_ROOT}" '
       f'--sequence {" ".join(DRIVES)} --eval-gt "{EVAL_GT}" --split "{CODE_DIR / SPLIT}" --part {PART} '
       f'--objects "{INPUT_ROOT}" --out "{folder(name, ph, spec)}" --qa {QA if not spec else 2} '
       f'--stage-dir "{STAGE_DIR / "frames"}" --threads {THREADS}' + (" --post-hoc-boxes" if ph else "")
       + (f' --perturb "{spec}"' if spec else "")
       + (" --save-preds" if SAVE_PREDS else "") + (f" --backbone {BACKBONE}" if BACKBONE else ""), cwd=CODE_DIR)

## Comparison

The same frames for every run. `ffr_pred` and `ffr_hazard` overall and in the near-object band; free-road IoU; the share of car and pedestrian box cells called free; calibration error. **FFR@R0.6 / R0.7**: `ffr_pred` at the confidence level where each model finds 60 % / 70 % of the true free road, so models are compared at equal recall (a plain `ffr_pred` favours a model that claims little); **AURC free**: risk-coverage area up to the ground truth's free share (lower is better).

In [ ]:
from IPython.display import Image, display

fmt = lambda v: "  n/a" if v is None else f"{v:.3f}"
rows = []
for name, ph, spec in EVALS:
    m = json.loads((folder(name, ph, spec) / "metrics.json").read_text())
    r, near = m["metrics"]["regions"]["all"], m["metrics"]["regions"]["near_object_2m"]
    cls = m["metrics"]["classes"]
    rows.append((label(name, ph, spec), m["mode"], m["checkpoint_epoch"] + 1, sum(m["frames"].values()), r["ffr_pred"],
                 r["ffr_hazard"], near["ffr_pred"], r["free_road"]["iou"], near["free_road"]["iou"],
                 cls["car"]["called_free"], cls["pedestrian"]["called_free"], m["metrics"]["calibration"]["ece"],
                 (m["metrics"].get("ffr_at_recall") or {}).get("0.6"), (m["metrics"].get("ffr_at_recall") or {}).get("0.7"),
                 m["metrics"].get("aurc_free"), (m.get("gate") or {}).get("mean_w_C")))
head = ("run", "mode", "epochs", "frames", "FFR pred", "FFR hazard", "FFR pred near", "free IoU", "free IoU near",
        "car called free", "ped called free", "ECE", "FFR@R0.6", "FFR@R0.7", "AURC free", "gate w_C")
print(" | ".join(head))
for row in rows:
    print(" | ".join(fmt(v) if v is None or isinstance(v, float) else str(v) for v in row))
for name, ph, spec in EVALS:
    for png in sorted((folder(name, ph, spec) / "qa").glob("*.png"))[:2 if not spec else 0]:
        display(Image(filename=str(png)))

## After the run

The output keeps `eval/<run>/<part>/` per run: `metrics.json` (all metrics with the run's config, checkpoint epoch and git commit), `frames.jsonl` (per-frame values for dispersion and failure cases) and the QA figures. Starter-phase numbers are for debugging; paper numbers come from the paper split (val 0006, test 0009 and 0010).